In [1]:
import pandas as pd

learn_data = pd.read_csv("minimal_train_fs.csv", header = None)
learn_data.columns = ['Age', 'TB', 'Alkphos', 'Sgot', 'ALB', 'AR', 'BilRatio', 'Female', 'Target']
learn_data["Female"] = learn_data["Female"].astype("category")
learn_data["Target"] = learn_data["Target"].astype("category")
learn_data.head()

,Age,TB,Alkphos,Sgot,ALB,AR,BilRatio,Female,Target
0,48,1.504077,5.641907,4.304065,2.4,0.52,0.511111,0,0
1,39,0.641854,5.192957,4.127134,4.3,1.38,0.473684,0,0
2,23,0.000000,5.356586,4.382027,3.1,1.00,0.300000,0,0
3,42,-0.356675,5.023881,4.394449,3.2,1.06,0.285714,1,0
4,54,3.117950,6.324359,3.610918,3.4,0.80,0.504425,1,0


In [2]:
from sklearn.model_selection import train_test_split

X = learn_data.drop(columns = ["Target"])
Xnum = X.drop(columns = ["Female"])
y = learn_data["Target"]

X_train, X_val, Xnum_train, Xnum_val, y_train, y_val = train_test_split(X, Xnum, y, test_size = 0.33, random_state = 42)

## Metrics

In [3]:
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score
import numpy as np

def compute_metrics (y_real, y_pred) -> list[float]:
    F1_macro = f1_score(y_real, y_pred, average = "macro")
    recall = recall_score(y_real, y_pred, average = "macro")
    prec = precision_score(y_real, y_pred, average = "macro")
    acc = accuracy_score(y_real, y_pred)
    return [F1_macro, recall, prec, acc]

def confusion (y_real, y_pred) -> None:
    TP = sum(np.logical_and(y_real == y_pred, y_real == 1))
    TN = sum(np.logical_and(y_real == y_pred, y_real == 0))
    FP = sum(np.logical_and(y_real != y_pred, y_real == 0))
    FN = sum(np.logical_and(y_real != y_pred, y_real == 1))
    print("\t\tPredicted")
    print("\t\t+1\t0")
    print(f"Real\t+1\t{TP}\t{FN}")
    print(f"\t0\t{FP}\t{TN}")
    print(f"Accuracy: {((TP + TN) / y_real.shape[0] * 100):.2f}%".format())

metrics_df = pd.DataFrame(columns = ["F1 Macro", "Recall", "Precision", "Accuracy"])

# KNN

In [6]:
from sklearn.neighbors import KNeighborsClassifier

knc = KNeighborsClassifier(n_neighbors = 5, weights = 'distance')
knc.fit(X_train, y_train)

confusion(np.array(y_train), pd.Series(knc.predict(X_train)))

		Predicted
		+1	0
Real	+1	84	0
	0	0	216
Accuracy: 100.00%


In [7]:
confusion(np.array(y_val), pd.Series(knc.predict(X_val)))

		Predicted
		+1	0
Real	+1	18	24
	0	19	88
Accuracy: 71.14%


In [49]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

knc = KNeighborsClassifier(metric = "minkowski", p = 3)
knc_pipeline = Pipeline([("scaler", StandardScaler()), ("knc", knc)])

n_neighbors = [2, 3, 5, 7, 10, 15]
distances = ["minkowski", "manhattan", "euclidean"]

knc_search = GridSearchCV(estimator = knc_pipeline,
                          param_grid = {"knc__n_neighbors" : n_neighbors,
                                        "knc__weights" : ("uniform", "distance"),
                                        "knc__metric" : distances},
                          scoring = "f1_macro",
                          cv = 5)
knc_search.fit(Xnum, y)
knc_search.best_params_

{'knc__metric': 'euclidean', 'knc__n_neighbors': 5, 'knc__weights': 'distance'}

In [50]:
knc_search.best_score_

0.6367674892387795

In [51]:
from sklearn.model_selection import cross_validate

knc_n_neighbors = knc_search.best_params_["knc__n_neighbors"]
knc_metric = knc_search.best_params_["knc__metric"]
knc_weights = knc_search.best_params_["knc__weights"]
knc_best = KNeighborsClassifier(n_neighbors = knc_n_neighbors,
                                metric = "minkowski", p = 3,
                                weights = knc_weights)
knc_pipeline = Pipeline([("scaler", StandardScaler()), ("knc", knc_best)])

cross_val_results = pd.DataFrame(cross_validate(knc_pipeline, Xnum, y, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

metrics_df.loc["KNN Minkowski", :] = mean_results
metrics_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
KNN Minkowski,0.615237,0.615399,0.619988,0.695031


## Predictions

In [52]:
test_data = pd.read_csv("minimal_test_fs.csv", header = None)
test_data.columns = ['Age', 'TB', 'Alkphos', 'Sgot', 'ALB', 'AR', 'BilRatio', 'Female']
test_data["Female"] = learn_data["Female"].astype("category")
test_data.head()

,Age,TB,Alkphos,Sgot,ALB,AR,BilRatio,Female
0,11,-0.356675,6.383507,3.367296,4.2,1.40,0.142857,0
1,62,0.587787,5.411646,5.043425,4.0,0.80,0.500000,0
2,60,-0.356675,5.159055,2.639057,4.2,1.10,0.285714,0
3,60,1.740466,5.365976,6.745236,3.2,0.78,0.491228,1
4,48,-0.105361,5.164786,3.988984,2.7,0.90,0.222222,1


In [53]:
test_data_num = test_data.drop(columns = ["Female"])

In [54]:
knc_pipeline.fit(Xnum, y)

labels_knc = pd.DataFrame(columns = ['ID', 'Label'])
labels_knc['Label'] = pd.DataFrame(knc_pipeline.predict(test_data_num))
labels_knc['ID'] = labels_knc.index + 1
labels_knc

,ID,Label
0,1,1
1,2,0
2,3,0
3,4,0
4,5,0
...,...,...
111,112,0
112,113,0
113,114,0
114,115,0


In [34]:
labels_knc.to_csv('new_predictions/knn_best.csv', index = False)